# VFX Production Intelligence Dashboard — Artists Cleaning

Work through this table from top to bottom. The context and rules below are inherited from earlier milestones.

## What we already know

- **Business name:** Artists
- **Purpose:** Contains records for artists used by the approved project analysis, including artist display name, primary VFX department, and production role.
- **One row represents:** One row per artist
- **Expected primary key:** `artist_id`

### Relationships
- manager_id → artists.artist_id

### Field rules from the approved data dictionary

| Field | Definition | Expected type | Null rule | Key role | Uniqueness | Allowed values / format | Cleaning expectation |
|---|---|---|---|---|---|---|---|
| `artist_id` | Unique identifier assigned to each artist record in the production workforce. | Identifier text | No nulls allowed | Primary key | Required | ART-###: uppercase ART, a hyphen, and three digits. | Remove exact duplicate artist rows, retain one canonical record per ID, trim whitespace, standardize casing, and verify that the cleaned key is unique and non-null. |
| `artist_name` | Full name used to identify the fictional artist in staffing and reporting views. | Text | No nulls allowed | Not a key | Not required | First and last name in consistent display casing. | Trim surrounding whitespace and standardize display casing without treating repeated names as duplicate artists. |
| `department` | Primary VFX production department to which the artist is assigned. | Category text | No nulls allowed | Not a key | Not required | Animation; Assets; Compositing; FX; Lighting; Matchmove; Matte Painting; Roto/Paint. | Trim whitespace and map all variants to the approved department labels. |
| `role` | Artist’s primary production role or discipline within the assigned department. | Category text | No nulls allowed | Not a key | Not required | Animator; Senior Animator; Compositor; Senior Compositor; FX Artist; Senior FX Artist; FX Lead; Lighting Artist; Senior Lighting Artist; Matchmove Artist; Matchmove Lead; Tracking Artist; Matte Painter; Modeler; Texture Artist; Paint Artist; Roto Artist. | Trim whitespace and preserve the approved production-role labels. |
| `seniority` | Artist experience level used for staffing, capacity, and labor-cost analysis. | Category text | No nulls allowed | Not a key | Not required | Junior; Mid; Senior; Lead. | Standardize capitalization to the approved seniority labels. |
| `location` | Artist’s assigned office location or remote-work region. | Category text | No nulls allowed | Not a key | Not required | Atlanta; London; Los Angeles; Montreal; Vancouver; Remote-US; Remote-EU. | Trim whitespace and standardize values to the approved location labels. |
| `weekly_capacity_hours` | Number of hours the artist is normally available for schedulable production work each week. | Decimal | No nulls allowed | Not a key | Not required | Non-negative numeric hours; zero is allowed only when the artist has no schedulable production capacity. | Remove text suffixes, convert to decimal, investigate negative and extreme values, and preserve zero only when supported by the artist’s current work status. |
| `hourly_cost_usd` | Fictional internal hourly labor-cost rate used to estimate artist and project labor cost. | Currency / decimal | No nulls allowed | Not a key | Not required | Positive numeric amount without currency symbols or text suffixes. | Remove formatting characters, convert to decimal, fill only business-confirmed missing values, and investigate or correct non-positive rates before cost analysis. |
| `hire_date` | Date the artist began employment with the fictional VFX studio. | Date | No nulls allowed | Not a key | Not required | YYYY-MM-DD. | Preserve as a date and validate that hire dates precede related production activity. |
| `active_flag` | Indicates whether the artist is currently active in the studio workforce. | Boolean | No nulls allowed | Not a key | Not required | Y = active; N = inactive. | Standardize affirmative values to Y and negative values to N; send unrecognized values for review. |
| `manager_id` | Identifier of the artist’s direct manager, department lead, or other reporting supervisor. | Identifier text | Conditionally nullable | Self-referencing foreign key | Not required | ART-###, or null when no manager is represented in the dataset. | Preserve legitimate top-level nulls, trim and standardize populated IDs, and validate every non-null value against artists.artist_id. |
| `email` | Synthetic, non-routable company email address assigned to the artist. | Text | No nulls allowed | Not a key | Required | first_name.last_name@spectraforgevfx.example. | Fill business-confirmed missing addresses, correct misassigned addresses, remove exact duplicate rows, and apply an approved unique-alias rule to unresolved same-name collisions. |

### Known issues and approved decisions
- artist_id: The raw table contains 62 rows and 60 distinct artist IDs; ART-044 and ART-053 each appear twice as exact duplicate records.
- artist_id: artist_id remains the intended primary key. The repeated IDs are raw-data defects that will be removed during cleaning.
- artist_name: Names are descriptive and may legitimately repeat; artist_id is the authoritative identifier.
- department: Raw values contain capitalization, abbreviation, and whitespace variants such as compositing, MATCHMOVE, matte painting, and Roto/Paint with trailing spaces.
- department: The raw text variants represent the same controlled department categories and will be standardized in the cleaned table.
- weekly_capacity_hours: The raw text field includes 40 hrs, one negative value (-8), one zero value, and one unusually high value (80).
- weekly_capacity_hours: The field is logically numeric even though dirty source values caused it to load as text. Invalid or unusual capacities will be corrected or documented before utilization analysis.
- hourly_cost_usd: The raw text field contains one blank, one negative value, a dollar-formatted value, and a value with /hr appended.
- hourly_cost_usd: The field is expected to be numeric currency; raw formatting, a blank, and a negative value explain the observed text type and require cleaning.
- active_flag: The raw field also contains Yes, Active, and FALSE.
- active_flag: The source stores one boolean concept with several text encodings. The cleaned field will use only Y and N.
- manager_id: Five raw records have null manager IDs; populated values must resolve to another artist record.
- manager_id: Nulls are valid only for artists with no manager represented in the dataset; populated values remain required to match an artist record.
- email: The raw field contains one missing address and duplicate or misassigned addresses, including duplicates caused by exact duplicate artist rows and same-name artists.
- email: Email is expected to be required and unique, but the raw export contains missing and repeated values that must be resolved during cleaning.


In [2]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path(r'C:/Users/Dan/Documents/MEGA/Dev/GitHub/career-accelerator/projects/project-01-vfx-production-intelligence')
TABLE_NAME = 'artists'
RAW_PATH = PROJECT_DIR / r'data/raw/csv/raw_artists.csv'
PROCESSED_PATH = PROJECT_DIR / r'data/processed/csv/artists.csv'
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)

if RAW_PATH.suffix.lower() == '.csv':
    raw_df = pd.read_csv(RAW_PATH)
elif RAW_PATH.suffix.lower() == '.parquet':
    raw_df = pd.read_parquet(RAW_PATH)
else:
    raise ValueError(f'Add the appropriate pandas reader for {RAW_PATH.suffix}')

clean_df = raw_df.copy()
print(f'{TABLE_NAME}: {len(raw_df):,} raw rows, {len(raw_df.columns)} columns')
raw_df.head()


artists: 62 raw rows, 12 columns


,artist_id,artist_name,department,role,seniority,location,weekly_capacity_hours,hourly_cost_usd,hire_date,active_flag,manager_id,email
0,ART-026,Morgan Wright,Animation,Animator,Mid,Remote-US,40,76,2020-04-13,Y,ART-006,morgan.wright@spectraforgevfx.example
1,ART-020,Finley King,Compositing,Compositor,Mid,Vancouver,40,61,2024-03-09,Yes,ART-015,finley.king@spectraforgevfx.example
2,ART-012,Nico Patel,Roto/Paint,Paint Artist,Senior,Remote-US,40,82,2018-03-03,Active,ART-033,nico.patel@spectraforgevfx.example
3,ART-028,Drew Thompson,Roto/Paint,Roto Artist,Mid,Los Angeles,40,73,2021-11-17,Y,ART-030,drew.thompson@spectraforgevfx.example
4,ART-024,Nico Allen,compositing,Senior Compositor,Senior,London,40,89,2025-09-30,Y,ART-019,nico.allen@spectraforgevfx.example


## 1. Profile the raw table

- Confirm the source row count and column names.
- Measure missing values by field.
- Check exact duplicate rows.
- Test uniqueness and nulls for `artist_id`.
- Review observed categories and parsing problems.
- Compare findings with the dictionary rules above before changing data.

In [3]:
# Write the profiling checks for this table here.
# Keep the outputs that justify your cleaning decisions.

%%sql

SELECT COLUMN_NAME
FROM raw_artists
WHERE TABLE_NAME = 'raw_artists';


SyntaxError: invalid syntax (1790929135.py, line 6)

## 2. Apply the approved cleaning plan

Transform `clean_df` without modifying `raw_df`. Follow the field-level expectations above. Document any treatment that differs from the approved dictionary.

In [ ]:
# Write this table's cleaning transformations here.
# Example structure only: clean_df = clean_df.copy()


## 3. Validate the processed result

- Required columns are still present.
- Expected logical types can be produced consistently.
- Required fields do not contain unresolved nulls.
- Allowed values and formats match the dictionary.
- Invalid negative, out-of-range, or impossible values are resolved or documented.
- `artist_id` is non-null and unique.
- Foreign-key and relationship exceptions are measured and documented.

In [ ]:
# Write the before-and-after validation checks here.
# The checks should fail visibly when an unresolved issue remains.


## 4. Export the reviewed table

After validation, save the reviewed result to `data/processed/csv/artists.csv`. The Data Cleaning Studio will discover and validate the file.

In [ ]:
# Run only after the table has passed your validation checks.
clean_df.to_csv(PROCESSED_PATH, index=False)
print(f'Saved {len(clean_df):,} rows to {PROCESSED_PATH}')


## Cleaning summary

<!-- Describe what changed, why each important decision was appropriate, how many records were affected, and any remaining exception that a later milestone must know about. -->
